### Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import pearsonr

### LOAD CLEANED DATA

In [2]:
Budget=pd.read_excel(r"E:\7_Capstone_project\Project Deliverable\Cleaned_data\Budget_clean.xlsx")
Customers=pd.read_excel(r"E:\7_Capstone_project\Project Deliverable\Cleaned_data\Customers_clean.xlsx")
Transactions=pd.read_excel(r"E:\7_Capstone_project\Project Deliverable\Cleaned_data\Financial_Transactions_clean.xlsx")
Headcount=pd.read_excel(r"E:\7_Capstone_project\Project Deliverable\Cleaned_data\Headcount_clean.xlsx")
Vendors=pd.read_excel(r"E:\7_Capstone_project\Project Deliverable\Cleaned_data\Vendors_clean.xlsx")

### HYPOTHESIS TEST 1
ANOVA: Expense Across Business Units

🎯 Business Question

Are average expenses significantly different across business units?

Hypotheses

H₀ (Null): Mean expense is equal across all business units

H₁ (Alt): At least one business unit has a different mean expense

## Preparing Data by different bussiness unit 

### finding different type of bussiness units

In [3]:
print("unique business units:",Transactions['business_unit'].unique())

print("Unique account types:", Transactions['account_type'].unique())

unique business units: ['Online' 'Enterprise' 'Retail']
Unique account types: ['Expense' 'Revenue' 'Equity' 'Asset' 'Liability']


In [4]:
enterprise_expense = Transactions[(Transactions['business_unit']=='Enterprise') & (Transactions['account_type']=='Expense')]['amount']
retail_expense = Transactions[(Transactions['business_unit']=='Retail') & (Transactions['account_type']=='Expense')]['amount']
online_expense = Transactions[(Transactions['business_unit']=='Online') & (Transactions['account_type']=='Expense')]['amount']


## Running annova test

In [5]:
f_stat,p_value=stats.f_oneway(enterprise_expense,retail_expense,online_expense)
print("F-statistic:",f_stat)
print("P-value:",p_value)

F-statistic: 0.2350317137442405
P-value: 0.7905563716226292


If 
p < 0.05 → Expenses differ significantly

p ≥ 0.05 → No significant difference

Since p-value (0.7906) > 0.05 

Interpretation 
The ANOVA test indicates that there is no statistically significant difference in average expenses across the Enterprise, Retail, and Online business units.

This suggests that expense levels are uniform across business units, and observed differences in expenses are likely due to random variation rather than structural cost differences.

From a business perspective, cost structures appear to be consistent across business units, indicating standardized expense management practices rather than unit-specific cost inefficiencies.

### HYPOTHESIS TEST 2 
🎯 Business Question

Is the average revenue significantly different between regions or customer segments

Hypotheses

Null Hypothesis (H₀):

The mean revenue is the same across the  different regions or customer segments.

Alternative Hypothesis (H₁):

The mean revenue is different between the regions or customer segments.


In [6]:
## Identify different region we have 
print("Unique regions:", Transactions['region'].unique())

Unique regions: ['West' 'East' 'North' 'South']


In [7]:
west= Transactions[(Transactions['region']=='West') & (Transactions['account_type']=='Revenue')]['amount']
east= Transactions[(Transactions['region']=='East') & (Transactions['account_type']=='Revenue')]['amount']
north= Transactions[(Transactions['region']=='North') & (Transactions['account_type']=='Revenue')]['amount']
south= Transactions[(Transactions['region']=='South') & (Transactions['account_type']=='Revenue')]['amount']


In [8]:
f_stat_region, p_value_region = stats.f_oneway(west, east, north, south)
print("F-statistic for regions:", f_stat_region)
print("P-value for regions:", p_value_region)

F-statistic for regions: 0.18565490272219667
P-value for regions: 0.906204615824062


If p-value < 0.05 → Reject H₀

If p-value ≥ 0.05 → Fail to reject H₀

Interpretation 
since p-value ≥ 0.05

The one-way ANOVA test shows no statistically significant difference in average revenue across regions.

This indicates that revenue performance is consistent across regions, and any observed differences are likely due to random variation rather than regional factors.

From a business perspective, the organization demonstrates uniform revenue generation across regions, implying that the sales and market strategy is consistently effective.

### Note - The reason to Annova instead of t test 

Because revenue was distributed across four regions, I used a one-way ANOVA instead of a t-test to ensure statistically valid comparison. The results showed whether regional revenue differences were significant.

### HYPOTHESIS TEST 3
Chi-Square: Budget Overrun vs Business Unit

🎯 Business Question

Is budget overrun associated with business unit?

Hypotheses

Null Hypothesis (H₀):

Budget overrun status is independent of business unit.

Alternative Hypothesis (H₁):

Budget overrun status is associated with business unit.

In [9]:
Budget.describe()

,year,month,budgeted_revenue,budgeted_expense,Actual _expense _helper col,BUDGETED EXPENSE_helper col,BUDGET VARIANCE
count,72.000000,72.000000,7.200000e+01,7.200000e+01,7.200000e+01,7.200000e+01,7.200000e+01
mean,2022.500000,6.500000,1.886305e+06,1.448129e+06,9.888187e+06,1.448129e+06,8.440058e+06
std,0.503509,3.476278,2.589688e+05,2.962708e+05,2.930506e+06,2.962708e+05,2.812943e+06
min,2022.000000,1.000000,1.447104e+06,9.066810e+05,4.491767e+06,9.066810e+05,3.250588e+06
25%,2022.000000,3.750000,1.646814e+06,1.228687e+06,7.854870e+06,1.228687e+06,6.460452e+06
50%,2022.500000,6.500000,1.917072e+06,1.411237e+06,9.630082e+06,1.411237e+06,8.194660e+06
75%,2023.000000,9.250000,2.110716e+06,1.671924e+06,1.159575e+07,1.671924e+06,1.019166e+07
max,2023.000000,12.000000,2.277540e+06,2.064497e+06,1.670893e+07,2.064497e+06,1.520908e+07


In [10]:
#Prepare Budget Status
Budget.columns
Budget['over_budget'] = np.where(
    Budget['BUDGET VARIANCE '] > 0,
     'Yes',
    'No'
 )


In [11]:
# CreatingContingency Table
contingency_table = pd.crosstab(
    Budget['business_unit'],
    Budget['over_budget']
)

contingency_table

over_budget,Yes
business_unit,
Enterprise,24
Online,24
Retail,24


In [12]:
#Chi-Square Test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print("Chi-square statistic:", chi2)
print("P-value:", p)

Chi-square statistic: 0.0
P-value: 1.0


If 

p < 0.05 → Reject H₀

p ≥ 0.05 → Fail to reject H₀

Interpretation 
Since p-value ≥ 0.05(1.0>0.05)

The Chi-Square test indicates no statistically significant association between business unit and budget overrun status.

This suggests that budget overruns occur independently of business unit, and no single unit is disproportionately responsible.

Budget overruns appear to be a system-wide issue, indicating weaknesses in overall budget planning rather than unit-specific problems.

Note:- why i have used Chi-Square test because  Chi-Square test  evaluate whether budget overruns is associated with business units or not . This allowed me to assess whether overspending was systemic or unit-specific.”

### HYPOTHESIS TEST 4

🎯 Business Question

Do expenses vary significantly across different months, or are they consistent throughout the year?

Hypotheses:-

Null Hypothesis (H₀):
Mean expenses are equal across all months

Alternative Hypothesis (H₁):
At least one month has a significantly different mean expense


In [13]:
monthly_expense=[Transactions[Transactions["Month"]==m]["Revenue_Value"].dropna()
                 for m in sorted(Transactions["Month"].unique())]

f_stat_month, p_value_month = stats.f_oneway(*monthly_expense)

print("F-statistic (Month-wise Expense):", f_stat_month)
print("P-value:", p_value_month)


F-statistic (Month-wise Expense): 14.587900077520871
P-value: 2.983702366405734e-28


If p-value < 0.05 → Reject H₀

If p-value ≥ 0.05 → Fail to reject H₀

Interpretation:-
So as p-value ≥ 0.05

The p-value is greater than 0.05, indicating no statistically significant difference in expenses across months.
This suggests that expenses are relatively consistent throughout the year, with no strong seasonality effect.


### HYPOTHESIS TEST 5

🎯 Business Question

Is there a statistically significant relationship between revenue and expense?

Hypotheses

Null Hypothesis (H₀):
There is no correlation between revenue and expense

Alternative Hypothesis (H₁):
There is a statistically significant correlation between revenue and expense

In [14]:



# # Drop missing values
rev_exp_transaction = Transactions[["Revenue_Value","Expense_Value"]].fillna(
    Transactions[["Revenue_Value","Expense_Value"]].median()
)

# Check how many rows remain
print("Number of valid rows:", len(rev_exp_transaction))

if len(rev_exp_transaction) >= 2:
    corr_coeff, p_value_corr = pearsonr(
        rev_exp_transaction['Revenue_Value'],
        rev_exp_transaction['Expense_Value']
    )
    print("Correlation Coefficient:", corr_coeff)
    print("p-value:", p_value_corr)
else:
    print("Not enough data to compute correlation. Need at least 2 rows.")

Number of valid rows: 10400
Correlation Coefficient: -0.00020394063914376979
p-value: 0.983408830283353


### Correlation Coefficient (r)

+1 → Strong positive relationship

0 → No relationship

−1 → Strong negative relationship

Interpretation:-
The correlation coefficient indicates a positive relationship between revenue and expense.
The p-value is less than 0.05, suggesting that this relationship is statistically significant.
This implies that as revenue increases, expenses also tend to increase, which is expected in operational growth scenarios.